In [0]:
taxi_df = spark.table("week2_silver_taxi_base")

display(taxi_df.limit(10))
taxi_df.printSchema()

In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    hour,
    dayofweek,
    month,
    year,
    unix_timestamp,
    round,
    when
)

In [0]:
taxi_time_df = (
    taxi_df
    .withColumn("pickup_date", to_date(col("pickup_datetime")))
    .withColumn("pickup_hour", hour(col("pickup_datetime")))
    .withColumn("pickup_day_of_week", dayofweek(col("pickup_datetime")))
    .withColumn("pickup_month", month(col("pickup_datetime")))
    .withColumn("pickup_year", year(col("pickup_datetime")))
    .withColumn(
        "trip_duration_minutes",
        round(
            (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60,
            2
        )
    )
)

In [0]:
taxi_time_df = taxi_time_df.withColumn(
    "pickup_time_band",
    when((col("pickup_hour") >= 6) & (col("pickup_hour") < 10), "Morning peak")
    .when((col("pickup_hour") >= 10) & (col("pickup_hour") < 16), "Daytime")
    .when((col("pickup_hour") >= 16) & (col("pickup_hour") < 20), "Evening peak")
    .otherwise("Night")
)

In [0]:
display(taxi_time_df.select(
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "pickup_year",
    "pickup_time_band",
    "trip_duration_minutes"
).limit(50))

In [0]:
taxi_time_df.printSchema()

In [0]:
taxi_time_df.write.mode("overwrite").saveAsTable("week2_silver_taxi_time_features")

In [0]:
display(spark.sql("SHOW TABLES LIKE 'week2_silver_taxi_time_features'"))

In [0]:
saved_time_df = spark.table("week2_silver_taxi_time_features")

display(saved_time_df.limit(10))
saved_time_df.printSchema()